# ***Ingeniería de características***

**Proyecto:** Detección, clasificación y medición de barras/anillos en galaxias  
**Equipo:** 39

## 1. Propósito

Este notebook crea la sección de **ingeniería de características** a partir de imágenes astronómicas en bandas **g, r, z**. La idea es preparar un dataset "final" que represente mejor la estructura visual de cada galaxia para después poder construir un modelo.

## 2. ¿Por qué transformamos con Lupton?

Las imágenes astronómicas suelen tener rangos de intensidad muy amplios: el centro de una galaxia puede ser muy brillante, mientras que brazos, barras o anillos pueden ser mucho más tenues. Una normalización lineal simple puede saturar el centro y ocultar estructuras débiles. La composición propuesta por **Lupton et al. (2004)** usa una transformación tipo *asinh* para comprimir intensidades altas y conservar información en regiones de bajo brillo.

En este notebook usamos Lupton con distintos parámetros `Q`, `stretch` y pesos por banda; lo cual nos permite aproximar dos objetivos:

- una imagen que sea más interpretable visualmente para revisión humana
- una representación que conserve patrones útiles para modelos de clasificación posteriores.

## 3. Ingeniería de características

El notebook genera distintas variables, agrupadas en:

- **estadísticos por banda:** media, desviación estándar, percentiles, IQR, rango robusto;
- **estructura radial:** intensidad central, zona interna, zona media, zona externa, contraste interno/externo y posible pico radial compatible con anillos;
- **textura y bordes:** entropía y respuesta promedio de bordes con Sobel;
- **asimetría:** comparación contra flips y rotación de 180°;
- **forma global:** centroide, momentos, elongación y elipticidad;
- **color relativo:** diferencias y razones entre bandas `z`, `r`, `g`;
- **features de la mejor imagen Lupton/RGB:** variables calculadas sobre la imagen visual final seleccionada.

Estas variables son interpretables y sirven como puente entre la imagen astronómica cruda, con el modelo posterior.

## 4. Salidas principales

La salida principal será el dataset "final", en formato .csv:

```text
/content/drive/MyDrive/ProyectoGalaxias/features/feature_engineering_lupton/final_engineered_features_lupton_selected.csv
```

También generamos:
- `selected_feature_ranking.csv`: ranking usado para escoger las variables finales;
- `selected_feature_dictionary.csv`: interpretación de las variables finales;
- `selected_feature_visualizations/`: gráficas de ranking, distribuciones y correlación;
- `best_transform_per_galaxy.csv`: mejor transformación visual por galaxia;
- `transformation_scores.csv`: score de cada transformación evaluada;
- `processed_images_lupton/`: PNG de la mejor imagen por galaxia;

## 5. Referencias

- [Lupton, R. et al. (2004). *Preparing Red-Green-Blue Images from CCD Data*. PASP.](https://arxiv.org/abs/astro-ph/0312483)
- [Astropy Visualization. `make_lupton_rgb`.](https://docs.astropy.org/en/latest/api/astropy.visualization.make_lupton_rgb.html)
- [Conselice, C. (2003). *The Relationship Between Stellar Light Distributions of Galaxies and their Formation Histories*.](https://arxiv.org/abs/astro-ph/0303065)
- [Lotz, J. et al. (2004). *A New Non-Parametric Approach to Galaxy Morphological Classification*.](https://arxiv.org/abs/astro-ph/0311352)


In [ ]:
# Dependencias de colab
import sys
import subprocess

packages = [
  "astropy",
  "scikit-image",
  "pillow",
  "tqdm",
  "scikit-learn"
]

for package in packages:
  subprocess.run(
      [sys.executable, "-m", "pip", "install", "-q", package],
      check=True
  )

In [ ]:
# Librerias y configuración
from pathlib import Path
import os
import subprocess
import re
import json
import math
import shutil
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.filters import sobel
from skimage.transform import resize
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import f_classif, mutual_info_classif
from tqdm.auto import tqdm
from PIL import Image
from astropy.io import fits
try:
  from astropy.visualization import make_lupton_rgb
  ASTROPY_LUPTON_AVAILABLE = True
except Exception:
  ASTROPY_LUPTON_AVAILABLE = False

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
print("Astropy Lupton disponible:", ASTROPY_LUPTON_AVAILABLE)

In [ ]:
# Conectar Google Drive y a GitHub
from google.colab import drive
drive.mount("/content/drive")

# Repositorio.
REPO_URL = "https://github.com/francoquintanilla0/RingGalaxiesAnalysis.git"
REPO_NAME = "RingGalaxiesAnalysis"
REPO_DIR = Path("/content") / REPO_NAME

# Clona el repositorio si no existe
if not (REPO_DIR / ".git").exists():
  print("Clonando repositorio...")
  subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
  print("Repositorio disponible.")

# Trabajamos en la carpeta del repositorio.
os.chdir(REPO_DIR)
print("Repositorio de trabajo:", REPO_DIR)
print("Carpeta actual:", Path.cwd())

In [ ]:
# Rutas
# Carpeta principal del proyecto en Google Drive.
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/ProyectoGalaxias")

# Carpeta donde están los datos pesados: FITS, NPY, catálogos, etc.
DATA_DIR = DRIVE_PROJECT_DIR / "data"

# Carpeta general de features.
FEATURES_DIR = DRIVE_PROJECT_DIR / "features"

# Carpeta específica de esta corrida.
OUTPUT_DIR = FEATURES_DIR / "feature_engineering_lupton"
PROCESSED_IMAGE_DIR = OUTPUT_DIR / "processed_images_lupton"
REPORT_DIR = OUTPUT_DIR / "reports"
SAMPLE_DIR = OUTPUT_DIR / "sample_visualizations"

for folder in [FEATURES_DIR, OUTPUT_DIR, PROCESSED_IMAGE_DIR, REPORT_DIR, SAMPLE_DIR]:
  folder.mkdir(parents=True, exist_ok=True)

# Raíces donde buscar catálogos e imágenes.
# Se prioriza Drive porque ahí viven los datos pesados del proyecto.
SEARCH_ROOTS = []
for root in [DATA_DIR, DRIVE_PROJECT_DIR, REPO_DIR]:
  if root.exists() and root not in SEARCH_ROOTS:
      SEARCH_ROOTS.append(root)

print("REPO_DIR:", REPO_DIR)
print("DRIVE_PROJECT_DIR:", DRIVE_PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("\nRaíces de búsqueda:")
for root in SEARCH_ROOTS:
  print("-", root)

if not DATA_DIR.exists():
  print("\nDATA_DIR no existe.")

In [ ]:
# Parámetros principales
MAX_IMAGES = None                  # None = procesa todas las imágenes. Para prueba rápida usar ejemplo (25)
RESUME_FROM_EXISTING = True        # Evita reprocesar galaxias ya guardadas en parciales.
SAVE_BEST_IMAGES = True            # Guarda una imagen PNG por galaxia con la mejor transformación.
SAVE_SAMPLE_GRIDS = True           # Guarda ejemplos visuales de transformaciones.
N_EXAMPLES = 10                    # Galaxias de ejemplo para visualizar.
BATCH_SAVE_EVERY = 100             # Guarda avances cada N galaxias.
IMAGE_MAX_SIZE = None              # None = no redimensiona. Ejemplo: 128 para acelerar.
TRANSFORM_MODE = "balanced"        # "balanced" recomendado. "full" prueba más combinaciones.

In [ ]:
# Funciones auxiliares
def should_exclude_path(path):
  """Evita leer salidas del propio notebook como si fueran datos de entrada."""
  p = Path(path)
  excluded_names = [
      "feature_engineering_lupton",
      "processed_images_lupton",
      "reports",
      ".git",
      "__pycache__"
  ]
  return any(name in p.parts for name in excluded_names)

def unique_paths(paths):
  """Remueve rutas duplicadas preservando el orden original."""
  seen = set()
  clean = []

  for p in paths:
      p = Path(p)
      try:
          key = str(p.resolve())
      except Exception:
          key = str(p)

      if key not in seen and not should_exclude_path(p):
          clean.append(p)
          seen.add(key)

  return clean

def find_files(patterns, roots):
  """Busca archivos con varios patrones dentro de una lista de carpetas raíz."""
  found = []

  for root in roots:
      root = Path(root)
      if not root.exists():
          continue

      for pattern in patterns:
          found.extend(root.rglob(pattern))

  return unique_paths(found)

def safe_relative(path, root=None):
  """Devuelve una ruta corta y legible para impresión en pantalla."""
  path = Path(path)
  root = root or REPO_DIR

  try:
      return str(path.relative_to(root))
  except Exception:
      return str(path)

def normalize_key(value):
  """Normaliza IDs y nombres de archivo para facilitar el match catálogo - imagen."""
  if value is None or (isinstance(value, float) and np.isnan(value)):
      return ""

  value = str(value).strip().lower()
  value = value.replace("\\", "/")
  value = value.split("/")[-1]

  # Remueve extensiones comunes.
  for ext in [".fits.gz", ".fits", ".fit", ".npy", ".npz", ".csv", ".png", ".jpg", ".jpeg"]:
      if value.endswith(ext):
          value = value[: -len(ext)]

  # Remueve sufijos típicos de banda o imagen compuesta.
  value = re.sub(r"(_|-)?(band)?[grz]$", "", value)
  value = re.sub(r"(_|-)?(grz|rgb|lupton|best|image|img)$", "", value)
  value = re.sub(r"[^a-z0-9]+", "_", value)
  value = re.sub(r"_+", "_", value).strip("_")

  return value

def numeric_key(value):
  """Extrae el último bloque numérico de un ID o nombre de archivo."""
  key = normalize_key(value)
  nums = re.findall(r"\d+", key)

  if len(nums) == 0:
      return ""

  return str(int(nums[-1])) if nums[-1].isdigit() else nums[-1]

def infer_band_from_filename(path):
  """Detecta si un archivo FITS corresponde a la banda g, r o z."""
  name = Path(path).name.lower()
  stem = name.replace(".fits.gz", "").replace(".fits", "").replace(".fit", "")

  patterns = {
      "g": [r"(^|[_\-])g($|[_\-])", r"bandg", r"_g$"],
      "r": [r"(^|[_\-])r($|[_\-])", r"bandr", r"_r$"],
      "z": [r"(^|[_\-])z($|[_\-])", r"bandz", r"_z$"],
  }

  for band, pats in patterns.items():
      for pat in pats:
          if re.search(pat, stem):
              return band

  return None

def read_csv_safely(path):
  """Lee catálogos CSV probando codificaciones comunes."""
  encodings = ["utf-8", "utf-8-sig", "latin1"]

  for enc in encodings:
      try:
          return pd.read_csv(path, encoding=enc, low_memory=False)
      except Exception:
          pass

  return pd.read_csv(path, low_memory=False)

## Carga de catálogos y construcción del catálogo de trabajo

En esta sección se localizan los catalogos/csv disponibles, el objetivo es:
- encontrar un identificador de galaxia;
- encontrar, si existe, una etiqueta o variable relacionada con anillos;
- generar una llave normalizada para empatar catálogo e imágenes.

In [ ]:
# Catalogo "maestro" de los csv principales
csv_files = find_files(["*.csv"], roots=SEARCH_ROOTS)

# Excluye outputs típicos de corridas previas.
csv_files = [
  p for p in csv_files
  if not any(x in p.name.lower() for x in [
      "features_raw_images",
      "transformation_scores",
      "best_transform",
      "final_engineered",
      "feature_selection",
      "work_catalog_lupton",
      "error_log"
  ])
]

print(f"CSVs candidatos encontrados: {len(csv_files)}")
for i, p in enumerate(csv_files[:30]):
  print(f"[{i}] {safe_relative(p, DRIVE_PROJECT_DIR)}")

ID_CANDIDATES = [
  "galaxy_id", "galaxy_id_fe", "image_id", "objid", "object_id", "id", "name",
  "iauname", "plateifu", "legacy_id"
]

LABEL_CANDIDATES = [
  "target_ring_internal", "label_binary", "label", "class", "clase",
  "anillos", "anillo", "ring", "ring_type", "tipo_anillo", "bar", "barra"
]

In [ ]:
# Funciones de las columnas de los csv
def infer_id_column(df):
  cols_lower = {c.lower(): c for c in df.columns}

  for c in ID_CANDIDATES:
      if c in cols_lower:
          return cols_lower[c]

  for c in df.columns:
      cl = c.lower()
      if "galaxy" in cl and "id" in cl:
          return c
      if cl.endswith("_id") or cl == "id":
          return c
      if "name" in cl:
          return c

  return None

def infer_label_columns(df):
  cols = []
  for c in df.columns:
      cl = c.lower()
      if cl in LABEL_CANDIDATES:
          cols.append(c)
      elif any(k in cl for k in ["anillo", "ring", "label", "class", "clase"]):
          cols.append(c)
  return cols

def score_catalog(path, df):
  """Asigna score para elegir el catálogo más útil."""
  name = path.name.lower()
  id_col = infer_id_column(df)
  label_cols = infer_label_columns(df)

  score = 0
  if id_col is not None:
      score += 5
  if len(label_cols) > 0:
      score += 5
  if any(k in name for k in ["catalog", "catalogo", "clas", "class", "anillo", "ring", "legacy", "imagenes"]):
      score += 3
  if any(c.lower() in ["ra", "dec", "redshift", "z"] for c in df.columns):
      score += 2
  if len(df) > 100:
      score += 1

  return score, id_col, label_cols

In [ ]:
# Catalogo
catalog_candidates = []
for path in csv_files:
  try:
      df_tmp = read_csv_safely(path)
      score, id_col, label_cols = score_catalog(path, df_tmp)
      catalog_candidates.append({
          "path": path,
          "rows": len(df_tmp),
          "cols": len(df_tmp.columns),
          "score": score,
          "id_col": id_col,
          "label_cols": label_cols,
          "df": df_tmp
      })
  except Exception as e:
      print("No se pudo leer:", path, "|", e)

catalog_candidates = sorted(catalog_candidates, key=lambda x: x["score"], reverse=True)

catalog_summary = pd.DataFrame([
  {
      "path": safe_relative(c["path"], DRIVE_PROJECT_DIR),
      "rows": c["rows"],
      "cols": c["cols"],
      "score": c["score"],
      "id_col": c["id_col"],
      "label_cols": ", ".join(c["label_cols"])
  }
  for c in catalog_candidates
])

display(catalog_summary.head(15))

if len(catalog_candidates) > 0 and catalog_candidates[0]["id_col"] is not None:
  primary_catalog_path = catalog_candidates[0]["path"]
  primary_catalog = catalog_candidates[0]["df"].copy()
  primary_id_col = catalog_candidates[0]["id_col"]
  primary_label_cols = catalog_candidates[0]["label_cols"]
  print("\nCatálogo principal seleccionado:")
  print(primary_catalog_path)
  print("ID:", primary_id_col)
  print("Labels candidatas:", primary_label_cols)
else:
  primary_catalog_path = None
  primary_catalog = pd.DataFrame()
  primary_id_col = None
  primary_label_cols = []
  print("\nNo se encontró un catálogo principal.")

In [ ]:
# Metadatos del catalogo y asignamos el taget que buscamos nosotros
def value_to_ring_target(value):
  """Convierte valores de catálogo a target binario cuando es posible.
  En donde:
  - 1 = anillo interno / presencia de anillo interno.
  - 0 = sin anillo.
  - NaN = no se puede inferir con seguridad.
  """
  if value is None or (isinstance(value, float) and np.isnan(value)):
      return np.nan

  # Numéricos: respeta 0/1 y considera 4/12 como anillos internos según el criterio del proyecto.
  if isinstance(value, (int, float, np.integer, np.floating)):
      if int(value) in [0, 1]:
          return int(value)
      if int(value) in [4, 12]:
          return 1
      return np.nan

  s = str(value).strip().lower()

  if s in ["", "nan", "none", "null"]:
      return np.nan

  # Casos de no anillo.
  negative_tokens = [
      "sin anillo", "sin_anillo", "no ring", "norings", "no_ring",
      "without ring", "none", "0", "false", "negativo", "negative"
  ]
  if any(tok in s for tok in negative_tokens):
      return 0

  # Casos de anillo interno.
  positive_tokens = [
      "anillo interno", "inner ring", "internal ring", "interno",
      "ring", "anillo", "4", "12", "true", "positivo", "positive"
  ]
  if any(tok in s for tok in positive_tokens):
      return 1

  # Si es número guardado como texto.
  try:
      num = int(float(s))
      if num in [0, 1]:
          return num
      if num in [4, 12]:
          return 1
  except Exception:
      pass

  return np.nan

def infer_target_from_row(row, label_cols):
  """Intenta inferir el target usando varias columnas candidatas."""
  for col in label_cols:
      if col in row.index:
          val = value_to_ring_target(row[col])
          if pd.notna(val):
              return val
  return np.nan

def build_catalog_metadata(df, id_col, label_cols, source_path=None):
  if df is None or len(df) == 0 or id_col is None:
      return pd.DataFrame()

  meta = df.copy()
  meta["catalog_id_original"] = meta[id_col].astype(str)
  meta["match_key"] = meta[id_col].apply(normalize_key)
  meta["match_num_key"] = meta[id_col].apply(numeric_key)

  # Target principal.
  if "target_ring_internal" not in meta.columns:
      meta["target_ring_internal"] = meta.apply(lambda r: infer_target_from_row(r, label_cols), axis=1)
  else:
      meta["target_ring_internal"] = meta["target_ring_internal"].apply(value_to_ring_target)

  meta["source_catalog"] = str(source_path) if source_path is not None else ""

  # Conserva columnas útiles sin volver gigante el dataframe.
  useful_cols = [
      "catalog_id_original", "match_key", "match_num_key", "target_ring_internal", "source_catalog",
      "ra", "dec", "redshift", "z", "label", "label_binary", "anillos", "anillo", "ring", "class", "clase"
  ]
  useful_cols = [c for c in useful_cols if c in meta.columns]

  return meta[useful_cols].drop_duplicates(subset=["match_key", "match_num_key"], keep="first")

catalog_meta = build_catalog_metadata(
  primary_catalog,
  primary_id_col,
  primary_label_cols,
  primary_catalog_path
)

print("Metadata de catálogo:", catalog_meta.shape)
display(catalog_meta.head())
display(catalog_meta.tail())

In [ ]:
# Indexar las imagenes de las galaxias
fits_files = find_files(["*.fits", "*.fit", "*.fits.gz"], roots=SEARCH_ROOTS)
npy_files = find_files(["*.npy", "*.npz"], roots=SEARCH_ROOTS)

print(f"FITS encontrados: {len(fits_files)}")
print(f"NPY/NPZ encontrados: {len(npy_files)}")

def clean_base_from_image_path(path):
  """Obtiene ID base desde nombre de archivo de imagen."""
  name = Path(path).name.lower()
  for ext in [".fits.gz", ".fits", ".fit", ".npy", ".npz"]:
      if name.endswith(ext):
          name = name[: -len(ext)]

  # Quita sufijo de banda o composición.
  name = re.sub(r"(_|-)?(band)?[grz]$", "", name)
  name = re.sub(r"(_|-)?(grz|rgb|lupton|image|img|best)$", "", name)
  return normalize_key(name)

image_records = {}

def get_record(base_key):
  if base_key not in image_records:
      image_records[base_key] = {
          "image_gid": base_key,
          "match_key": base_key,
          "match_num_key": numeric_key(base_key),
          "g_path": "",
          "r_path": "",
          "z_path": "",
          "grz_npy_path": "",
          "npz_path": "",
          "fits_multiband_path": "",
          "available_source": ""
      }
  return image_records[base_key]

# FITS: agrupar por banda g/r/z.
for path in fits_files:
  band = infer_band_from_filename(path)
  base_key = clean_base_from_image_path(path)
  rec = get_record(base_key)

  if band in ["g", "r", "z"]:
      rec[f"{band}_path"] = str(path)
      rec["available_source"] = "fits_bands"
  else:
      # Si no se identifica banda, puede ser un FITS multibanda.
      if rec["fits_multiband_path"] == "":
          rec["fits_multiband_path"] = str(path)
          rec["available_source"] = "fits_multiband"

# NPY/NPZ: es decir el GRZ compuesto.
for path in npy_files:
  base_key = clean_base_from_image_path(path)
  rec = get_record(base_key)

  if path.suffix.lower() == ".npz":
      rec["npz_path"] = str(path)
      rec["available_source"] = "npz_grz"
  else:
      rec["grz_npy_path"] = str(path)
      rec["available_source"] = "npy_grz"

image_index = pd.DataFrame(list(image_records.values()))

if len(image_index) > 0:
  image_index["has_complete_fits_bands"] = (
      image_index["g_path"].astype(str).str.len().gt(0)
      & image_index["r_path"].astype(str).str.len().gt(0)
      & image_index["z_path"].astype(str).str.len().gt(0)
  )
  image_index["has_npy_grz"] = image_index["grz_npy_path"].astype(str).str.len().gt(0)
  image_index["has_npz"] = image_index["npz_path"].astype(str).str.len().gt(0)
  image_index["has_fits_multiband"] = image_index["fits_multiband_path"].astype(str).str.len().gt(0)

  image_index["is_processable"] = (
      image_index["has_complete_fits_bands"]
      | image_index["has_npy_grz"]
      | image_index["has_npz"]
      | image_index["has_fits_multiband"]
  )

  image_index = image_index.sort_values(["is_processable", "image_gid"], ascending=[False, True]).reset_index(drop=True)
else:
  image_index = pd.DataFrame(columns=[
      "image_gid", "match_key", "match_num_key", "g_path", "r_path", "z_path",
      "grz_npy_path", "npz_path", "fits_multiband_path", "available_source",
      "has_complete_fits_bands", "has_npy_grz", "has_npz", "has_fits_multiband", "is_processable"
  ])

print("Índice de imágenes:", image_index.shape)
print("Procesables:", int(image_index["is_processable"].sum()) if len(image_index) else 0)
display(image_index.head())
image_index.to_csv(OUTPUT_DIR / "image_index_lupton.csv", index=False)

In [ ]:
# Construir el catalogo master
processable_images = image_index[image_index["is_processable"]].copy()

if len(catalog_meta) > 0 and len(processable_images) > 0:
  # Match 1: por llave normalizada.
  work_catalog = processable_images.merge(
      catalog_meta,
      on="match_key",
      how="left",
      suffixes=("", "_cat")
  )

  # Match 2: para los no encontrados, intenta por último bloque numérico.
  if "catalog_id_original" in work_catalog.columns:
      unmatched_mask = work_catalog["catalog_id_original"].isna()
  else:
      unmatched_mask = pd.Series([True] * len(work_catalog), index=work_catalog.index)

  if unmatched_mask.any():
      catalog_by_num = catalog_meta[
          catalog_meta["match_num_key"].astype(str).str.len().gt(0)
      ].drop_duplicates("match_num_key", keep="first")

      fallback = processable_images.loc[unmatched_mask, processable_images.columns].merge(
          catalog_by_num.drop(columns=["match_key"], errors="ignore"),
          on="match_num_key",
          how="left",
          suffixes=("", "_num")
      )

      for col in fallback.columns:
          if col not in work_catalog.columns:
              work_catalog[col] = np.nan

      work_catalog.loc[unmatched_mask, fallback.columns] = fallback.values

else:
  work_catalog = processable_images.copy()
  work_catalog["target_ring_internal"] = np.nan
  work_catalog["catalog_id_original"] = ""

# ID final para este notebook.
if len(work_catalog) > 0:
  work_catalog["galaxy_id_fe"] = work_catalog["image_gid"].astype(str)
  work_catalog = work_catalog.drop_duplicates("galaxy_id_fe", keep="first").reset_index(drop=True)

  if MAX_IMAGES is not None:
      work_catalog = work_catalog.head(int(MAX_IMAGES)).copy()

# Guarda catálogo.
work_catalog_path = OUTPUT_DIR / "work_catalog_lupton.csv"
work_catalog.to_csv(work_catalog_path, index=False)

print("Catálogo de trabajo:", work_catalog.shape)
print("Con target:", int(work_catalog["target_ring_internal"].notna().sum()) if "target_ring_internal" in work_catalog.columns else 0)
print("Guardado en:", work_catalog_path)

display(work_catalog.head())

if len(work_catalog) == 0:
  print("\nAVISO: no se encontraron imágenes procesables.")

## Normalización de imágenes

Aquí se preparan las imágenes para que todas puedan tratarse de forma homogénea. Leémos imágenes separadas por banda (`g`, `r`, `z`) o arreglos multibanda ya guardados como `.npy`/`.npz`.

La convención que se usa después para formar RGB es:

`R = z`, `G = r`, `B = g`

Antes de aplicar transformaciones normalizamos las intensidades por percentiles.


In [ ]:
# Funciones de carga de las imagenes y normalización de intensidades
def sanitize_image(img):
  """Convierte una imagen a float32 y reemplaza valores no finitos."""
  arr = np.asarray(img, dtype=np.float32)
  if arr.size == 0:
      return arr

  finite = np.isfinite(arr)
  if not finite.any():
      return np.zeros_like(arr, dtype=np.float32)

  med = np.nanmedian(arr[finite])
  arr = np.where(np.isfinite(arr), arr, med).astype(np.float32)
  return arr

def ensure_2d(arr):
  """Convierte arreglos FITS/NPY a 2D cuando es posible."""
  arr = np.asarray(arr)
  arr = np.squeeze(arr)

  if arr.ndim == 2:
      return arr.astype(np.float32)

  if arr.ndim == 3:
      # Si es 3xHxW o HxWx3, se maneja en split_grz_array.
      # Como fallback, toma el primer plano.
      if arr.shape[0] not in [3, 4] and arr.shape[-1] in [3, 4]:
          return arr[..., 0].astype(np.float32)
      return arr[0].astype(np.float32)

  raise ValueError(f"No se pudo convertir a 2D. Shape recibido: {arr.shape}")

def split_grz_array(arr):
  """Separa el arreglo multibanda en z, r, g.
  Convención del proyecto:
  - canal 0 = z
  - canal 1 = r
  - canal 2 = g
  """
  arr = np.asarray(arr)
  arr = np.squeeze(arr)

  if arr.ndim == 2:
      img = sanitize_image(arr)
      return {"z": img, "r": img, "g": img}

  if arr.ndim != 3:
      raise ValueError(f"Arreglo multibanda no válido. Shape: {arr.shape}")

  if arr.shape[-1] >= 3:
      z = arr[..., 0]
      r = arr[..., 1]
      g = arr[..., 2]
  elif arr.shape[0] >= 3:
      z = arr[0]
      r = arr[1]
      g = arr[2]
  else:
      raise ValueError(f"No se detectaron 3 bandas. Shape: {arr.shape}")

  return {
      "z": sanitize_image(z),
      "r": sanitize_image(r),
      "g": sanitize_image(g),
  }

def read_fits_image(path):
  """Lee la primera extensión FITS que contenga datos."""
  path = Path(path)
  if not path.exists():
      raise FileNotFoundError(path)

  with fits.open(path, memmap=False) as hdul:
      data = None
      for hdu in hdul:
          if getattr(hdu, "data", None) is not None:
              data = hdu.data
              break

  if data is None:
      raise ValueError(f"FITS sin datos: {path}")

  return np.asarray(data)

def read_npy_or_npz(path):
  """Lee NPY o NPZ y devuelve el arreglo principal."""
  path = Path(path)
  if not path.exists():
      raise FileNotFoundError(path)

  if path.suffix.lower() == ".npz":
      data = np.load(path)
      keys = list(data.keys())
      if len(keys) == 0:
          raise ValueError(f"NPZ vacío: {path}")
      return data[keys[0]]

  return np.load(path)

def maybe_resize_image(img, max_size=IMAGE_MAX_SIZE):
  """Redimensiona si se define IMAGE_MAX_SIZE."""
  img = sanitize_image(img)

  if max_size is None:
      return img

  h, w = img.shape
  max_dim = max(h, w)

  if max_dim <= max_size:
      return img

  scale = max_size / max_dim
  new_h = max(8, int(round(h * scale)))
  new_w = max(8, int(round(w * scale)))

  return resize(
      img,
      (new_h, new_w),
      preserve_range=True,
      anti_aliasing=True
  ).astype(np.float32)


def load_bands(row):
  """Carga bandas g/r/z desde NPY/NPZ, FITS separados o FITS multibanda."""
  # Prioridad 1: NPY GRZ.
  npy_path = str(row.get("grz_npy_path", "") or "")
  if len(npy_path) > 0 and Path(npy_path).exists():
      arr = read_npy_or_npz(npy_path)
      bands = split_grz_array(arr)
      return {k: maybe_resize_image(v) for k, v in bands.items()}

  # Prioridad 2: NPZ.
  npz_path = str(row.get("npz_path", "") or "")
  if len(npz_path) > 0 and Path(npz_path).exists():
      arr = read_npy_or_npz(npz_path)
      bands = split_grz_array(arr)
      return {k: maybe_resize_image(v) for k, v in bands.items()}

  # Prioridad 3: FITS separados.
  g_path = str(row.get("g_path", "") or "")
  r_path = str(row.get("r_path", "") or "")
  z_path = str(row.get("z_path", "") or "")

  if all(Path(p).exists() for p in [g_path, r_path, z_path]):
      bands = {
          "g": sanitize_image(ensure_2d(read_fits_image(g_path))),
          "r": sanitize_image(ensure_2d(read_fits_image(r_path))),
          "z": sanitize_image(ensure_2d(read_fits_image(z_path))),
      }
      return {k: maybe_resize_image(v) for k, v in bands.items()}

  # Prioridad 4: FITS multibanda.
  multi_path = str(row.get("fits_multiband_path", "") or "")
  if len(multi_path) > 0 and Path(multi_path).exists():
      arr = read_fits_image(multi_path)
      bands = split_grz_array(arr)
      return {k: maybe_resize_image(v) for k, v in bands.items()}

  raise FileNotFoundError(f"No se encontraron imágenes para {row.get('galaxy_id_fe', 'sin_id')}")

def percentile_scale(img, p_low=1, p_high=99, eps=1e-8):
  """Escala robusta a [0,1] usando percentiles."""
  arr = sanitize_image(img)
  lo, hi = np.nanpercentile(arr, [p_low, p_high])
  if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < eps:
      return np.zeros_like(arr, dtype=np.float32)
  scaled = (arr - lo) / (hi - lo + eps)
  return np.clip(scaled, 0, 1).astype(np.float32)

def normalize_band(img, mode="p1p99"):
  """Normalización por banda para transformación visual."""
  if mode == "p05p995":
      return percentile_scale(img, 0.5, 99.5)
  if mode == "p1p99":
      return percentile_scale(img, 1, 99)
  if mode == "p2p98":
      return percentile_scale(img, 2, 98)
  if mode == "zscore":
      arr = sanitize_image(img)
      med = np.nanmedian(arr)
      mad = np.nanmedian(np.abs(arr - med)) + 1e-8
      z = (arr - med) / (1.4826 * mad)
      return np.clip((z + 3) / 6, 0, 1).astype(np.float32)

  return percentile_scale(img, 1, 99)

## Transformaciones: Lupton, asinh, log y sqrt

La transformación principal es **Lupton RGB**, porque permite comprimir rangos altos de intensidad y conservar estructura en zonas débiles. En este notebook se prueban variantes con:

- `Q`: controla qué tan agresiva es la compresión tipo asinh;
- `stretch`: controla la escala de brillo antes de la compresión;
- `weights`: permite dar más o menos peso a cada banda.

Además se comparan transformaciones `asinh`, `log` y `sqrt` como alternativas visuales. La mejor transformación no se elige “a ojo” manualmente, sino con un score heurístico que combina contraste, entropía, bordes, saturación, balance de color y contraste radial.

In [ ]:
# Transformaciones
WEIGHT_PROFILES = {
  # Balance base.
  "equal": {"z": 1.00, "r": 1.00, "g": 1.00},

  # Da un poco más de fuerza a z, útil si estructuras internas destacan en rojo/infrarrojo.
  "z_plus": {"z": 1.20, "r": 1.00, "g": 0.90},

  # Da un poco más de fuerza a r, útil para estructura estelar de brillo medio.
  "r_plus": {"z": 1.00, "r": 1.20, "g": 0.90},

  # Más suave en g para evitar que ruido azul.
  "warm_soft": {"z": 1.15, "r": 1.05, "g": 0.80},
}

def stack_rgb_from_channels(red, green, blue):
  rgb = np.dstack([
      np.clip(red, 0, 1),
      np.clip(green, 0, 1),
      np.clip(blue, 0, 1)
  ]).astype(np.float32)
  return rgb

def lupton_fallback(red, green, blue, Q=8, stretch=0.8, eps=1e-8):
  """Implementación simple tipo Lupton si Astropy falla."""
  red = np.clip(red, 0, None)
  green = np.clip(green, 0, None)
  blue = np.clip(blue, 0, None)

  intensity = (red + green + blue) / 3.0
  x = Q * intensity / (stretch + eps)
  factor = np.ones_like(intensity, dtype=np.float32)
  mask = x > eps
  factor[mask] = np.arcsinh(x[mask]) / (x[mask] + eps)
  rgb = stack_rgb_from_channels(red * factor, green * factor, blue * factor)

  # Reescala global para aprovechar rango visual.
  hi = np.nanpercentile(rgb, 99.5)
  if np.isfinite(hi) and hi > eps:
      rgb = np.clip(rgb / hi, 0, 1)
  return rgb.astype(np.float32)

def transform_lupton_rgb(bands, Q=8, stretch=0.8, weights=None, scale_mode="p1p99"):
  """Composición Lupton con z,r,g a rojo,verde,azul."""
  weights = weights or WEIGHT_PROFILES["equal"]

  # Colores
  red = normalize_band(bands["z"], scale_mode) * weights.get("z", 1.0)
  green = normalize_band(bands["r"], scale_mode) * weights.get("r", 1.0)
  blue = normalize_band(bands["g"], scale_mode) * weights.get("g", 1.0)

  if ASTROPY_LUPTON_AVAILABLE:
      try:
          rgb = make_lupton_rgb(
              red,
              green,
              blue,
              Q=Q,
              stretch=stretch,
              minimum=0
          )
          rgb = np.asarray(rgb).astype(np.float32)
          if rgb.max() > 1:
              rgb = rgb / 255.0
          return np.clip(rgb, 0, 1)
      except Exception:
          return lupton_fallback(red, green, blue, Q=Q, stretch=stretch)

  return lupton_fallback(red, green, blue, Q=Q, stretch=stretch)

def transform_asinh_rgb(bands, a=8, weights=None, scale_mode="p1p99"):
  """Transformación con arcoseno hiperbolico"""
  weights = weights or WEIGHT_PROFILES["equal"]

  z = normalize_band(bands["z"], scale_mode) * weights.get("z", 1.0)
  r = normalize_band(bands["r"], scale_mode) * weights.get("r", 1.0)
  g = normalize_band(bands["g"], scale_mode) * weights.get("g", 1.0)

  z = np.arcsinh(a * z) / np.arcsinh(a)
  r = np.arcsinh(a * r) / np.arcsinh(a)
  g = np.arcsinh(a * g) / np.arcsinh(a)

  return stack_rgb_from_channels(z, r, g)

def transform_log_rgb(bands, a=50, weights=None, scale_mode="p1p99"):
  """Transformación con logaritmo"""
  weights = weights or WEIGHT_PROFILES["equal"]

  z = normalize_band(bands["z"], scale_mode) * weights.get("z", 1.0)
  r = normalize_band(bands["r"], scale_mode) * weights.get("r", 1.0)
  g = normalize_band(bands["g"], scale_mode) * weights.get("g", 1.0)

  z = np.log1p(a * z) / np.log1p(a)
  r = np.log1p(a * r) / np.log1p(a)
  g = np.log1p(a * g) / np.log1p(a)

  return stack_rgb_from_channels(z, r, g)

def transform_sqrt_rgb(bands, weights=None, scale_mode="p1p99"):
  """Transformación con raiz cuadrada"""
  weights = weights or WEIGHT_PROFILES["equal"]

  z = np.sqrt(normalize_band(bands["z"], scale_mode) * weights.get("z", 1.0))
  r = np.sqrt(normalize_band(bands["r"], scale_mode) * weights.get("r", 1.0))
  g = np.sqrt(normalize_band(bands["g"], scale_mode) * weights.get("g", 1.0))

  return stack_rgb_from_channels(z, r, g)

def build_transform_grid(mode="balanced"):
  """Combinaciones interpretables de parámetros visuales."""
  grid = []

  if mode == "full":
      Q_values = [4, 6, 8, 10, 12, 16]
      stretch_values = [0.3, 0.5, 0.8, 1.2, 2.0]
      scale_modes = ["p05p995", "p1p99", "p2p98"]
      weight_names = list(WEIGHT_PROFILES.keys())
  else:
      Q_values = [4, 8, 12]
      stretch_values = [0.4, 0.8, 1.2]
      scale_modes = ["p1p99"]
      weight_names = ["equal", "z_plus", "r_plus", "warm_soft"]

  for Q in Q_values:
      for stretch in stretch_values:
          for scale_mode in scale_modes:
              for weight_name in weight_names:
                  grid.append({
                      "name": f"lupton_Q{Q}_s{stretch}_{scale_mode}_{weight_name}",
                      "family": "lupton",
                      "Q": Q,
                      "stretch": stretch,
                      "scale_mode": scale_mode,
                      "weight_profile": weight_name
                  })

  # Transformaciones comparativas no-Lupton.
  for a in [4, 8, 16]:
      grid.append({
          "name": f"asinh_a{a}_p1p99_equal",
          "family": "asinh",
          "a": a,
          "scale_mode": "p1p99",
          "weight_profile": "equal"
      })

  for a in [25, 75]:
      grid.append({
          "name": f"log_a{a}_p1p99_equal",
          "family": "log",
          "a": a,
          "scale_mode": "p1p99",
          "weight_profile": "equal"
      })

  grid.append({
      "name": "sqrt_p1p99_equal",
      "family": "sqrt",
      "scale_mode": "p1p99",
      "weight_profile": "equal"
  })

  return grid

TRANSFORM_GRID = build_transform_grid(TRANSFORM_MODE)
print("Número de transformaciones:", len(TRANSFORM_GRID))
display(pd.DataFrame(TRANSFORM_GRID).head(12))

## Métricas visuales e ingeniería de características

A partir de las bandas crudas y de la mejor imagen RGB se calculan variables.
Algunas ideas clave que tenemos que tener en cuenta, son:

- **Perfil radial:** resume cómo cambia el brillo desde el centro hacia el exterior. Es útil porque un anillo puede verse como un incremento de intensidad en una zona intermedia.
- **Concentración central:** compara brillo del centro contra zonas externas. Ayuda a distinguir galaxias dominadas por bulbo/núcleo de galaxias con estructura más distribuida.
- **Contraste interno/externo:** mide si la zona interna destaca sobre el borde exterior.
- **Pico radial de anillo:** busca el radio donde aparece un máximo local fuera del núcleo extremo; no confirma por sí solo un anillo, pero puede ser una señal candidata.
- **Entropía y bordes:** capturan riqueza visual y estructura local.
- **Asimetría:** compara la imagen contra reflejos y rotación de 180°, inspirada en enfoques morfológicos usados en astronomía.
- **Momentos y elipticidad:** describen orientación, elongación y distribución global de la luz.
- **Color entre bandas:** compara `z`, `r` y `g` para capturar diferencias de intensidad por longitud de onda.

In [ ]:
# Métricas visuales y características
def image_entropy(img, bins=128):
  """Entropía normalizada aproximada de una imagen en [0,1]."""
  x = np.asarray(img, dtype=np.float32)
  x = np.clip(x, 0, 1).ravel()
  if x.size == 0:
      return 0.0

  hist, _ = np.histogram(x, bins=bins, range=(0, 1), density=False)
  prob = hist.astype(np.float64)
  prob = prob / (prob.sum() + 1e-12)
  prob = prob[prob > 0]
  ent = -np.sum(prob * np.log2(prob))
  return float(ent / np.log2(bins))

def radial_profile(image2d, n_bins=40):
  """Perfil radial promedio desde el centro de la imagen."""
  img = np.asarray(image2d, dtype=np.float32)
  img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

  h, w = img.shape
  cy, cx = (h - 1) / 2, (w - 1) / 2

  y, x = np.indices(img.shape)
  r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
  r_norm = r / (r.max() + 1e-8)

  bins = np.linspace(0, 1, n_bins + 1)
  profile = np.zeros(n_bins, dtype=np.float32)

  for i in range(n_bins):
      mask = (r_norm >= bins[i]) & (r_norm < bins[i + 1])
      if mask.any():
          profile[i] = float(np.nanmean(img[mask]))
      else:
          profile[i] = np.nan
  return profile, bins[:-1]

def radial_features(img, prefix):
  """Features radiales enfocadas en concentración y posibles anillos."""
  img = percentile_scale(img, 1, 99)
  prof, rbins = radial_profile(img, n_bins=40)

  center = np.nanmean(prof[:4])
  inner = np.nanmean(prof[4:10])
  mid = np.nanmean(prof[10:24])
  outer = np.nanmean(prof[24:])

  # Zona candidata para anillo: evita el centro extremo y el borde.
  candidate = prof[6:30]
  if np.isfinite(candidate).any():
      peak_local = int(np.nanargmax(candidate))
      peak_idx = peak_local + 6
      peak_val = float(prof[peak_idx])
      peak_radius = float(rbins[peak_idx])
  else:
      peak_idx = -1
      peak_val = np.nan
      peak_radius = np.nan

  base = np.nanmedian(np.r_[prof[2:6], prof[30:36]]) if len(prof) >= 36 else np.nanmedian(prof)
  ring_contrast = (peak_val - base) / (abs(base) + 1e-8) if np.isfinite(peak_val) else np.nan

  # Pendiente radial simple centro - fuera.
  valid = np.isfinite(prof)
  if valid.sum() >= 3:
      slope = float(np.polyfit(rbins[valid], prof[valid], 1)[0])
  else:
      slope = np.nan
  return {
      f"{prefix}_radial_center_mean": float(center),
      f"{prefix}_radial_inner_mean": float(inner),
      f"{prefix}_radial_mid_mean": float(mid),
      f"{prefix}_radial_outer_mean": float(outer),
      f"{prefix}_central_concentration": float(center / (outer + 1e-8)),
      f"{prefix}_inner_outer_contrast": float((inner - outer) / (abs(outer) + 1e-8)),
      f"{prefix}_ring_peak_value": float(peak_val),
      f"{prefix}_ring_peak_radius": float(peak_radius),
      f"{prefix}_ring_peak_contrast": float(ring_contrast),
      f"{prefix}_radial_slope": float(slope),
  }

def asymmetry_features(img, prefix):
  """Asimetría respecto a flips y rotación de 180 grados."""
  x = percentile_scale(img, 1, 99)

  flip_h = np.flip(x, axis=1)
  flip_v = np.flip(x, axis=0)
  rot = np.rot90(x, 2)

  denom = np.sum(np.abs(x)) + 1e-8
  return {
      f"{prefix}_asym_horizontal": float(np.sum(np.abs(x - flip_h)) / denom),
      f"{prefix}_asym_vertical": float(np.sum(np.abs(x - flip_v)) / denom),
      f"{prefix}_asym_rot180": float(np.sum(np.abs(x - rot)) / denom),
  }

def moment_features(img, prefix):
  """Momentos simples de forma/elongación usando intensidad positiva."""
  x = percentile_scale(img, 1, 99)
  h, w = x.shape

  y, xx = np.indices(x.shape)
  weights = np.clip(x, 0, None)
  total = weights.sum() + 1e-8

  cx = float((xx * weights).sum() / total)
  cy = float((y * weights).sum() / total)

  x0 = xx - cx
  y0 = y - cy

  mu20 = float((weights * x0 ** 2).sum() / total)
  mu02 = float((weights * y0 ** 2).sum() / total)
  mu11 = float((weights * x0 * y0).sum() / total)

  cov = np.array([[mu20, mu11], [mu11, mu02]], dtype=np.float64)
  eigvals = np.linalg.eigvalsh(cov)
  eigvals = np.sort(np.maximum(eigvals, 1e-8))[::-1]

  elongation = float(np.sqrt(eigvals[0] / eigvals[1]))
  ellipticity = float(1 - np.sqrt(eigvals[1] / eigvals[0]))
  return {
      f"{prefix}_centroid_x_norm": float(cx / max(w - 1, 1)),
      f"{prefix}_centroid_y_norm": float(cy / max(h - 1, 1)),
      f"{prefix}_moment_mu20": mu20,
      f"{prefix}_moment_mu02": mu02,
      f"{prefix}_moment_mu11": mu11,
      f"{prefix}_elongation": elongation,
      f"{prefix}_ellipticity": ellipticity,
  }

def band_stat_features(img, prefix):
  """Estadísticos robustos por imagen/banda."""
  x = sanitize_image(img)
  p = np.nanpercentile(x, [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100])
  scaled = percentile_scale(x, 1, 99)

  feats = {
      f"{prefix}_mean": float(np.nanmean(x)),
      f"{prefix}_std": float(np.nanstd(x)),
      f"{prefix}_min": float(p[0]),
      f"{prefix}_p01": float(p[1]),
      f"{prefix}_p05": float(p[2]),
      f"{prefix}_p10": float(p[3]),
      f"{prefix}_p25": float(p[4]),
      f"{prefix}_p50": float(p[5]),
      f"{prefix}_p75": float(p[6]),
      f"{prefix}_p90": float(p[7]),
      f"{prefix}_p95": float(p[8]),
      f"{prefix}_p99": float(p[9]),
      f"{prefix}_max": float(p[10]),
      f"{prefix}_iqr": float(p[6] - p[4]),
      f"{prefix}_robust_range_p99_p01": float(p[9] - p[1]),
      f"{prefix}_positive_fraction": float(np.mean(x > 0)),
      f"{prefix}_zero_fraction": float(np.mean(x == 0)),
      f"{prefix}_entropy": image_entropy(scaled),
      f"{prefix}_edge_mean": float(np.mean(sobel(scaled))),
      f"{prefix}_edge_std": float(np.std(sobel(scaled))),
  }

  feats.update(radial_features(x, prefix))
  feats.update(asymmetry_features(x, prefix))
  feats.update(moment_features(x, prefix))
  return feats

def color_features(bands):
  """Relaciones de color/intensidad entre bandas z, r, g."""
  feats = {}
  stats = {}
  for b in ["z", "r", "g"]:
      x = sanitize_image(bands[b])
      stats[b] = {
          "mean": float(np.nanmean(x)),
          "median": float(np.nanmedian(x)),
          "p95": float(np.nanpercentile(x, 95)),
          "std": float(np.nanstd(x)),
      }

  for a, b in [("z", "r"), ("z", "g"), ("r", "g")]:
      for stat in ["mean", "median", "p95", "std"]:
          va = stats[a][stat]
          vb = stats[b][stat]
          feats[f"color_{a}_minus_{b}_{stat}"] = float(va - vb)
          feats[f"color_{a}_ratio_{b}_{stat}"] = float(va / (abs(vb) + 1e-8))

  # Luminancia simple con composición z,r,g.
  lum = (
      percentile_scale(bands["z"], 1, 99)
      + percentile_scale(bands["r"], 1, 99)
      + percentile_scale(bands["g"], 1, 99)
  ) / 3.0

  feats.update(band_stat_features(lum, "luminance"))
  return feats

def visual_quality_metrics(rgb):
  """Score --heurístico-- para elegir la imagen que mejor conserva estructura visual.
  Buscamos:
  - contraste razonable
  - buena entropía
  - bordes visibles
  - baja saturación
  - balance de color
  - contraste radial
  """
  rgb = np.asarray(rgb, dtype=np.float32)
  rgb = np.clip(rgb, 0, 1)

  lum = rgb.mean(axis=2)
  edge = sobel(lum)

  contrast = float(np.nanstd(lum))
  entropy = image_entropy(lum)
  edge_mean = float(np.nanmean(edge))
  saturation_fraction = float(np.mean((rgb <= 0.01) | (rgb >= 0.99)))

  channel_means = rgb.reshape(-1, 3).mean(axis=0)
  color_balance = float(1.0 - (np.std(channel_means) / (np.mean(channel_means) + 1e-8)))
  color_balance = float(np.clip(color_balance, 0, 1))

  rf = radial_features(lum, "tmp")
  ring_contrast = float(rf["tmp_ring_peak_contrast"]) if np.isfinite(rf["tmp_ring_peak_contrast"]) else 0.0

  # Penaliza saturación excesiva y premia estructura/contraste.
  score = (
      0.30 * np.clip(contrast / 0.25, 0, 1)
      + 0.25 * np.clip(entropy, 0, 1)
      + 0.20 * np.clip(edge_mean / 0.15, 0, 1)
      + 0.15 * np.clip(abs(ring_contrast) / 1.5, 0, 1)
      + 0.10 * color_balance
      - 0.25 * np.clip(saturation_fraction / 0.35, 0, 1)
  )
  return {
      "visual_score": float(score),
      "visual_contrast": contrast,
      "visual_entropy": float(entropy),
      "visual_edge_mean": edge_mean,
      "visual_saturation_fraction": saturation_fraction,
      "visual_color_balance": color_balance,
      "visual_ring_contrast": ring_contrast,
  }

def apply_transform(bands, spec):
  family = spec["family"]
  weights = WEIGHT_PROFILES.get(spec.get("weight_profile", "equal"), WEIGHT_PROFILES["equal"])
  scale_mode = spec.get("scale_mode", "p1p99")

  if family == "lupton":
      return transform_lupton_rgb(
          bands,
          Q=spec.get("Q", 8),
          stretch=spec.get("stretch", 0.8),
          weights=weights,
          scale_mode=scale_mode
      )

  if family == "asinh":
      return transform_asinh_rgb(
          bands,
          a=spec.get("a", 8),
          weights=weights,
          scale_mode=scale_mode
      )

  if family == "log":
      return transform_log_rgb(
          bands,
          a=spec.get("a", 50),
          weights=weights,
          scale_mode=scale_mode
      )

  if family == "sqrt":
      return transform_sqrt_rgb(
          bands,
          weights=weights,
          scale_mode=scale_mode
      )
  raise ValueError(f"Familia de transformación no reconocida: {family}")

def extract_all_features_from_bands(bands):
  """Extrae features desde bandas crudas y luminancia."""
  feats = {}
  for b in ["g", "r", "z"]:
      feats.update(band_stat_features(bands[b], f"raw_{b}"))

  feats.update(color_features(bands))
  return feats

def extract_features_from_best_rgb(rgb):
  """Extrae features desde la imagen RGB seleccionada."""
  rgb = np.asarray(rgb, dtype=np.float32)
  lum = rgb.mean(axis=2)
  feats = {}
  feats.update(band_stat_features(lum, "best_rgb_luminance"))
  for i, channel in enumerate(["red_z", "green_r", "blue_g"]):
      feats.update(band_stat_features(rgb[..., i], f"best_rgb_{channel}"))
  return feats

In [ ]:
# Visualización de ejemplos de las transformaciones
def show_transform_examples(work_catalog, n_examples=N_EXAMPLES):
  if len(work_catalog) == 0:
      print("No hay imágenes.")
      return
  examples = work_catalog.sample(
      n=min(n_examples, len(work_catalog)),
      random_state=RANDOM_STATE
  ).reset_index(drop=True)

  for _, row in examples.iterrows():
      galaxy_id = str(row["galaxy_id_fe"])
      try:
          bands = load_bands(row)
          rgb_linear = stack_rgb_from_channels(
              normalize_band(bands["z"], "p1p99"),
              normalize_band(bands["r"], "p1p99"),
              normalize_band(bands["g"], "p1p99")
          )

          rgb_lupton = transform_lupton_rgb(
              bands,
              Q=8,
              stretch=0.8,
              weights=WEIGHT_PROFILES["equal"],
              scale_mode="p1p99"
          )

          rgb_lupton_z = transform_lupton_rgb(
              bands,
              Q=8,
              stretch=0.8,
              weights=WEIGHT_PROFILES["z_plus"],
              scale_mode="p1p99"
          )

          rgb_asinh = transform_asinh_rgb(
              bands,
              a=8,
              weights=WEIGHT_PROFILES["equal"],
              scale_mode="p1p99"
          )

          # Plots
          fig, axes = plt.subplots(1, 8, figsize=(24, 3))

          axes[0].imshow(percentile_scale(bands["g"], 1, 99), cmap="gray", origin="lower")
          axes[0].set_title("Banda g")

          axes[1].imshow(percentile_scale(bands["r"], 1, 99), cmap="gray", origin="lower")
          axes[1].set_title("Banda r")

          axes[2].imshow(percentile_scale(bands["z"], 1, 99), cmap="gray", origin="lower")
          axes[2].set_title("Banda z")

          axes[3].imshow(rgb_linear, origin="lower")
          axes[3].set_title("RGB percentil")

          axes[4].imshow(rgb_lupton, origin="lower")
          axes[4].set_title("Lupton base")

          axes[5].imshow(rgb_lupton_z, origin="lower")
          axes[5].set_title("Lupton z+")

          axes[6].imshow(rgb_asinh, origin="lower")
          axes[6].set_title("asinh")

          # Perfil radial de luminancia Lupton.
          lum = rgb_lupton.mean(axis=2)
          prof, rbins = radial_profile(lum, n_bins=40)
          axes[7].plot(rbins, prof)
          axes[7].set_title("Perfil radial")
          axes[7].set_xlabel("Radio normalizado")
          axes[7].set_ylabel("Intensidad")

          target = row.get("target_ring_internal", np.nan)
          fig.suptitle(f"galaxy_id={galaxy_id} | target_ring_internal={target}", y=1.08)

          for ax in axes[:7]:
              ax.axis("off")

          plt.tight_layout()

          if SAVE_SAMPLE_GRIDS:
              out_path = SAMPLE_DIR / f"sample_{galaxy_id}.png"
              fig.savefig(out_path, dpi=130, bbox_inches="tight")
          plt.show()

      except Exception as e:
          print(f"Error visualizando {galaxy_id}: {e}")

# Llamamos la función
show_transform_examples(work_catalog, n_examples=N_EXAMPLES)

## Procesamiento

En esta sección se procesan TODAS galaxias del catálogo. Para cada una:
1. Creamos sus bandas `g`, `r`, `z`
2. Probamos con las transformaciones definidas
3. Calculamos un score visual para cada transformación
4. Seleccionamos la mejor transformación
5. Extraemos features desde bandas crudas y desde la imagen RGB seleccionada
6. Guardamos los resultados parciales para poder retomar la corrida sin perder avances


In [ ]:
# Procesamiento de las features y la mejor por galaxia
FEATURES_PARTIAL_PATH = OUTPUT_DIR / "features_raw_images_partial.csv"
SCORES_PARTIAL_PATH = OUTPUT_DIR / "transformation_scores_partial.csv"
BEST_PARTIAL_PATH = OUTPUT_DIR / "best_transform_per_galaxy_partial.csv"
ERROR_LOG_PATH = OUTPUT_DIR / "error_log_lupton.csv"
FEATURES_FINAL_PATH = OUTPUT_DIR / "features_raw_images.csv"
SCORES_FINAL_PATH = OUTPUT_DIR / "transformation_scores.csv"
BEST_FINAL_PATH = OUTPUT_DIR / "best_transform_per_galaxy.csv"

def safe_filename(text):
  text = str(text)
  text = re.sub(r"[^A-Za-z0-9_\-]+", "_", text)
  return text[:180]

def save_rgb_image(rgb, path):
  arr = np.clip(np.asarray(rgb) * 255, 0, 255).astype(np.uint8)
  Image.fromarray(arr).save(path)

def evaluate_transformations_for_bands(bands):
  """Evalúa todas las transformaciones y devuelve mejor resultado con base en la tabla de scores."""
  local_scores = []
  best_score = -np.inf
  best_spec = None
  best_rgb = None
  best_metrics = None

  for spec in TRANSFORM_GRID:
      try:
          rgb = apply_transform(bands, spec)
          metrics = visual_quality_metrics(rgb)

          row = {
              "transform_name": spec["name"],
              "transform_family": spec["family"],
              "Q": spec.get("Q", np.nan),
              "stretch": spec.get("stretch", np.nan),
              "a": spec.get("a", np.nan),
              "scale_mode": spec.get("scale_mode", ""),
              "weight_profile": spec.get("weight_profile", ""),
              **metrics
          }
          local_scores.append(row)

          if metrics["visual_score"] > best_score:
              best_score = metrics["visual_score"]
              best_spec = spec
              best_rgb = rgb
              best_metrics = metrics

      except Exception as e:
          local_scores.append({
              "transform_name": spec.get("name", "unknown"),
              "transform_family": spec.get("family", "unknown"),
              "error": str(e)
          })

  return best_spec, best_rgb, best_metrics, local_scores

def load_existing_partial(path, id_col="galaxy_id_fe"):
  if RESUME_FROM_EXISTING and path.exists():
      try:
          df = pd.read_csv(path, low_memory=False)
          if id_col in df.columns:
              return df, set(df[id_col].astype(str))
          return df, set()
      except Exception:
          return pd.DataFrame(), set()
  return pd.DataFrame(), set()

existing_features_df, processed_ids = load_existing_partial(FEATURES_PARTIAL_PATH, "galaxy_id_fe")
print(f"Galaxias ya procesadas: {len(processed_ids)}")

feature_rows = []
score_rows = []
best_rows = []
error_rows = []

if len(work_catalog) == 0:
  print("No hay imágenes para procesar.")
else:
  iterator = tqdm(work_catalog.iterrows(), total=len(work_catalog), desc="Procesando galaxias")

  for i, (_, row) in enumerate(iterator, start=1):
      galaxy_id = str(row["galaxy_id_fe"])

      if RESUME_FROM_EXISTING and galaxy_id in processed_ids:
          continue

      try:
          bands = load_bands(row)

          # 1) Features crudas por banda.
          features = {
              "galaxy_id_fe": galaxy_id,
              "image_gid": row.get("image_gid", ""),
              "available_source": row.get("available_source", ""),
              "target_ring_internal": row.get("target_ring_internal", np.nan),
              "catalog_id_original": row.get("catalog_id_original", ""),
              "ra": row.get("ra", np.nan),
              "dec": row.get("dec", np.nan),
              "redshift": row.get("redshift", np.nan),
              "z_catalog": row.get("z", np.nan),
          }
          features.update(extract_all_features_from_bands(bands))

          # 2) Evaluación de transformaciones visuales.
          best_spec, best_rgb, best_metrics, local_scores = evaluate_transformations_for_bands(bands)

          for srow in local_scores:
              srow["galaxy_id_fe"] = galaxy_id
              score_rows.append(srow)

          # 3) Features de la mejor imagen realzada.
          if best_rgb is not None and best_spec is not None:
              best_features = extract_features_from_best_rgb(best_rgb)
              features.update(best_features)
              best_summary = {
                  "galaxy_id_fe": galaxy_id,
                  "best_transform_name": best_spec.get("name", ""),
                  "best_transform_family": best_spec.get("family", ""),
                  "best_Q": best_spec.get("Q", np.nan),
                  "best_stretch": best_spec.get("stretch", np.nan),
                  "best_a": best_spec.get("a", np.nan),
                  "best_scale_mode": best_spec.get("scale_mode", ""),
                  "best_weight_profile": best_spec.get("weight_profile", ""),
                  **{f"best_{k}": v for k, v in best_metrics.items()},
              }
              best_rows.append(best_summary)
              features.update(best_summary)

              if SAVE_BEST_IMAGES:
                  out_name = f"{safe_filename(galaxy_id)}__{safe_filename(best_spec.get('name', 'best'))}.png"
                  out_path = PROCESSED_IMAGE_DIR / out_name
                  save_rgb_image(best_rgb, out_path)
                  features["best_image_path"] = str(out_path)
                  best_rows[-1]["best_image_path"] = str(out_path)
          feature_rows.append(features)

      except Exception as e:
          error_rows.append({
              "galaxy_id_fe": galaxy_id,
              "error": str(e)
          })

      # Guardado parcial por batches.
      if i % BATCH_SAVE_EVERY == 0:
          partial_features = pd.concat([existing_features_df, pd.DataFrame(feature_rows)], ignore_index=True)
          if len(partial_features) > 0:
              partial_features = partial_features.drop_duplicates("galaxy_id_fe", keep="last")
              partial_features.to_csv(FEATURES_PARTIAL_PATH, index=False)

          pd.DataFrame(score_rows).to_csv(SCORES_PARTIAL_PATH, index=False)
          pd.DataFrame(best_rows).to_csv(BEST_PARTIAL_PATH, index=False)
          pd.DataFrame(error_rows).to_csv(ERROR_LOG_PATH, index=False)

# Consolidado final.
features_df = pd.concat([existing_features_df, pd.DataFrame(feature_rows)], ignore_index=True)
if len(features_df) > 0 and "galaxy_id_fe" in features_df.columns:
  features_df = features_df.drop_duplicates("galaxy_id_fe", keep="last")

scores_df = pd.DataFrame(score_rows)
best_df = pd.DataFrame(best_rows)
errors_df = pd.DataFrame(error_rows)

# Caracteristicas parciales
features_df.to_csv(FEATURES_PARTIAL_PATH, index=False)
scores_df.to_csv(SCORES_PARTIAL_PATH, index=False)
best_df.to_csv(BEST_PARTIAL_PATH, index=False)
errors_df.to_csv(ERROR_LOG_PATH, index=False)

# Caracteristicas finales
features_df.to_csv(FEATURES_FINAL_PATH, index=False)
scores_df.to_csv(SCORES_FINAL_PATH, index=False)
best_df.to_csv(BEST_FINAL_PATH, index=False)

print("Procesamiento finalizado.")
print("Features:", features_df.shape, ", guardado en: ", FEATURES_FINAL_PATH)
print("Scores:", scores_df.shape, ", guardado en: ", SCORES_FINAL_PATH)
print("Best:", best_df.shape, ", guardado en: ", BEST_FINAL_PATH)
print("Errores:", errors_df.shape, ", guardado en: ", ERROR_LOG_PATH)

display(features_df.head())
display(best_df.head())
display(errors_df.head())

## Dataset final para modelado futuro.

1. El archivo final `final_engineered_features_lupton_selected.csv` es nuestro dataset final.
2. El dataset final contiene como **máximo 20 features** más columnas mínimas de identificación y target
3. Las variables se seleccionan con:
   - Dominio: variables radiales, contraste de anillo, concentración, textura, asimetría y color
   - Estadísticas: relación con `target_ring_internal`, cuando el target está disponible.


In [ ]:
# Creación de nuestro dataset final -- con las variables más importantes
FINAL_SELECTED_PATH = OUTPUT_DIR / "final_engineered_features_lupton_selected.csv"
SELECTION_REPORT_PATH = REPORT_DIR / "selected_feature_ranking.csv"
SELECTED_DICTIONARY_PATH = REPORT_DIR / "selected_feature_dictionary.csv"
MAX_FINAL_FEATURES = 20

# Columnas que sí queremos conservar aunque no sean variables predictoras.
ID_COLUMNS_FINAL = ["galaxy_id_fe", "image_gid", "catalog_id_original", "best_image_path"]
TARGET_COLUMN = "target_ring_internal"

if "features_df" not in globals() or len(features_df) == 0:
  if FEATURES_FINAL_PATH.exists():
      features_df = pd.read_csv(FEATURES_FINAL_PATH, low_memory=False)
  else:
      features_df = pd.DataFrame()

base_df = features_df.copy()

if len(base_df) == 0:
  print("No hay features.")
else:
  # Variables con mejor sentido para anillos internos.
  mandatory_features = [
      "best_visual_score",
      "luminance_ring_peak_contrast",
      "luminance_inner_outer_contrast",
      "luminance_central_concentration",
      "luminance_entropy",
      "luminance_edge_mean",
      "luminance_asym_rot180",
      "luminance_ellipticity",
      "raw_z_ring_peak_contrast",
      "raw_r_ring_peak_contrast",
  ]

  candidate_features = [
      # Calidad visual de la mejor transformación.
      "best_visual_score",
      "best_visual_contrast",
      "best_visual_entropy",
      "best_visual_edge_mean",
      "best_visual_ring_contrast",
      "best_visual_saturation_fraction",
      "best_visual_color_balance",

      # Luminancia global z/r/g: estructura general de la galaxia.
      "luminance_ring_peak_contrast",
      "luminance_ring_peak_radius",
      "luminance_inner_outer_contrast",
      "luminance_central_concentration",
      "luminance_entropy",
      "luminance_edge_mean",
      "luminance_asym_rot180",
      "luminance_elongation",
      "luminance_ellipticity",

      # Señales radiales por banda.
      "raw_z_ring_peak_contrast",
      "raw_r_ring_peak_contrast",
      "raw_g_ring_peak_contrast",
      "raw_z_central_concentration",
      "raw_r_central_concentration",
      "raw_g_central_concentration",
      "raw_z_inner_outer_contrast",
      "raw_r_inner_outer_contrast",
      "raw_g_inner_outer_contrast",

      # Textura/bordes por banda.
      "raw_z_entropy",
      "raw_r_entropy",
      "raw_g_entropy",
      "raw_z_edge_mean",
      "raw_r_edge_mean",
      "raw_g_edge_mean",

      # Forma y simetría.
      "raw_z_asym_rot180",
      "raw_r_asym_rot180",
      "raw_g_asym_rot180",
      "raw_z_ellipticity",
      "raw_r_ellipticity",
      "raw_g_ellipticity",

      # Color relativo entre las bandas.
      "color_z_minus_r_mean",
      "color_z_ratio_r_mean",
      "color_z_minus_g_mean",
      "color_z_ratio_g_mean",
      "color_r_minus_g_mean",
      "color_r_ratio_g_mean",
      "color_z_minus_r_p95",
      "color_z_ratio_r_p95",
  ]

  # Mantiene solo columnas candidatas y las obligatorias
  candidate_features = [c for c in candidate_features if c in base_df.columns]
  mandatory_features = [c for c in mandatory_features if c in base_df.columns]

  # Solo features numéricas.
  numeric_candidates = []
  for col in candidate_features:
      base_df[col] = pd.to_numeric(base_df[col], errors="coerce")
      if base_df[col].notna().sum() > 0:
          numeric_candidates.append(col)

# Limpieza
  if TARGET_COLUMN in base_df.columns:
      base_df[TARGET_COLUMN] = pd.to_numeric(base_df[TARGET_COLUMN], errors="coerce")
  clean_features = base_df[numeric_candidates].replace([np.inf, -np.inf], np.nan)

  # Imputación simple con mediana.
  medians = clean_features.median(numeric_only=True)
  clean_features = clean_features.fillna(medians).fillna(0)

  # Elimina columnas constantes o casi constantes.
  variances = clean_features.var(axis=0)
  numeric_candidates = variances[variances > 1e-12].index.tolist()
  clean_features = clean_features[numeric_candidates]

# Ranking de las mejores por relevancia
  ranking_rows = []
  target_available = (
      TARGET_COLUMN in base_df.columns
      and base_df[TARGET_COLUMN].notna().sum() > 0
      and base_df[TARGET_COLUMN].nunique(dropna=True) >= 2
  )

  if target_available and len(numeric_candidates) > 0:
      valid_mask = base_df[TARGET_COLUMN].notna()
      X_rank = clean_features.loc[valid_mask, numeric_candidates]
      y_rank = base_df.loc[valid_mask, TARGET_COLUMN].astype(int)

      # F-score ANOVA: relación lineal/univariada con la clase.
      f_scores, p_values = f_classif(X_rank, y_rank)

      # Relación no lineal aproximada con la clase.
      mi_scores = mutual_info_classif(
          X_rank,
          y_rank,
          random_state=RANDOM_STATE,
          discrete_features=False
      )

      f_scores = np.nan_to_num(f_scores, nan=0.0, posinf=0.0, neginf=0.0)
      mi_scores = np.nan_to_num(mi_scores, nan=0.0, posinf=0.0, neginf=0.0)

      f_norm = f_scores / (f_scores.max() + 1e-12)
      mi_norm = mi_scores / (mi_scores.max() + 1e-12)

      for i, col in enumerate(numeric_candidates):
          domain_priority = 1.0 if col in mandatory_features else 0.5

          # Score combinado, usando:
          # - 55% información mutua,
          # - 35% F-score,
          # - 10% prioridad metodológica.
          selection_score = (
              0.55 * mi_norm[i]
              + 0.35 * f_norm[i]
              + 0.10 * domain_priority
          )

          ranking_rows.append({
              "feature": col,
              "f_score": float(f_scores[i]),
              "p_value": float(p_values[i]) if np.isfinite(p_values[i]) else np.nan,
              "mutual_info": float(mi_scores[i]),
              "domain_priority": domain_priority,
              "selection_score": float(selection_score),
              "selection_reason": "dominio + relación estadística con el target"
          })
  else:
      # Si no hay target, se usa únicamente prioridad de dominio y dominio.
      for col in numeric_candidates:
          domain_priority = 1.0 if col in mandatory_features else 0.5
          ranking_rows.append({
              "feature": col,
              "f_score": np.nan,
              "p_value": np.nan,
              "mutual_info": np.nan,
              "domain_priority": domain_priority,
              "selection_score": float(domain_priority),
              "selection_reason": "selección por criterio de dominio; target no disponible"
          })

  selected_feature_ranking = pd.DataFrame(ranking_rows)

  if len(selected_feature_ranking) > 0:
      selected_feature_ranking = selected_feature_ranking.sort_values(
          ["selection_score", "domain_priority"],
          ascending=False
      ).reset_index(drop=True)
  else:
      selected_feature_ranking = pd.DataFrame(columns=[
          "feature", "f_score", "p_value", "mutual_info",
          "domain_priority", "selection_score", "selection_reason"
      ])

# Guardamos solamente el num de variables que nos interesan
  selected_features = []

  # Primero asegura variables metodológicamente clave.
  for col in mandatory_features:
      if col in numeric_candidates and col not in selected_features:
          selected_features.append(col)

  # Luego completa con las mejores por ranking.
  for col in selected_feature_ranking["feature"].tolist():
      if col not in selected_features:
          selected_features.append(col)
      if len(selected_features) >= MAX_FINAL_FEATURES:
          break

  selected_features = selected_features[:MAX_FINAL_FEATURES]

  if len(selected_feature_ranking) > 0:
      selected_feature_ranking["selected"] = selected_feature_ranking["feature"].isin(selected_features)
      selected_feature_ranking["selected_rank"] = np.where(
          selected_feature_ranking["selected"],
          selected_feature_ranking["feature"].map({f: i + 1 for i, f in enumerate(selected_features)}),
          np.nan
      )

# df final completo
  id_cols_existing = [c for c in ID_COLUMNS_FINAL if c in base_df.columns]
  final_cols = id_cols_existing.copy()

  if TARGET_COLUMN in base_df.columns:
      final_cols.append(TARGET_COLUMN)

  final_cols.extend(selected_features)

  # Evita duplicados conservando el orden.
  final_cols = list(dict.fromkeys(final_cols))

  final_selected_df = pd.concat(
      [
          base_df[[c for c in final_cols if c not in selected_features]],
          clean_features[selected_features]
      ],
      axis=1
  )

  final_selected_df.to_csv(FINAL_SELECTED_PATH, index=False, encoding="utf-8-sig")
  selected_feature_ranking.to_csv(SELECTION_REPORT_PATH, index=False, encoding="utf-8-sig")

  print("Dataset final guardado en:")
  print(FINAL_SELECTED_PATH)

  print("Variables seleccionadas:")
  for i, feat in enumerate(selected_features, start=1):
      print(f"{i:02d}. {feat}")

  display(final_selected_df.head())

## Features seleccionadas

Después de construir el dataset final, queremos revisar cómo se comportan las variables elegidas.

1. **ranking de relevancia** de las features seleccionadas
2. **distribuciones por clase** (`sin anillo` vs `anillo interno`), si el target está disponible
3. **matriz de correlación**, para detectar variables correlacionadas

In [ ]:
# Features seleccionadas
FEATURE_VIZ_DIR = REPORT_DIR / "selected_feature_visualizations"
FEATURE_VIZ_DIR.mkdir(parents=True, exist_ok=True)

if "final_selected_df" not in globals() or len(final_selected_df) == 0:
  if FINAL_SELECTED_PATH.exists():
      final_selected_df = pd.read_csv(FINAL_SELECTED_PATH, low_memory=False)
  else:
      final_selected_df = pd.DataFrame()

if "selected_features" not in globals():
  selected_features = [
      c for c in final_selected_df.columns
      if c not in ID_COLUMNS_FINAL + [TARGET_COLUMN]
  ]

if len(final_selected_df) == 0 or len(selected_features) == 0:
  print("No hay dataset final o features seleccionadas para visualizar.")
else:
  # Ranking de las fetures
  if "selected_feature_ranking" in globals() and len(selected_feature_ranking) > 0:
      ranking_plot_df = selected_feature_ranking[
          selected_feature_ranking["feature"].isin(selected_features)
      ].copy()

      ranking_plot_df = ranking_plot_df.sort_values("selection_score", ascending=True)

      plt.figure(figsize=(10, max(4, 0.35 * len(ranking_plot_df))))
      plt.barh(ranking_plot_df["feature"], ranking_plot_df["selection_score"])
      plt.xlabel("Score combinado de selección")
      plt.title("Features seleccionadas: ranking de relevancia")
      plt.tight_layout()

      ranking_plot_path = FEATURE_VIZ_DIR / "selected_features_ranking.png"
      plt.savefig(ranking_plot_path, dpi=150, bbox_inches="tight")
      plt.show()

      display(
          selected_feature_ranking[selected_feature_ranking["selected"]]
          .sort_values("selected_rank")
          .reset_index(drop=True)
      )

  # Distribución por clase
  target_ok = (
      TARGET_COLUMN in final_selected_df.columns
      and final_selected_df[TARGET_COLUMN].notna().sum() > 0
      and final_selected_df[TARGET_COLUMN].nunique(dropna=True) >= 2
  )

  features_to_plot = selected_features[:min(8, len(selected_features))]

  if target_ok:
      for feature in features_to_plot:
          x0 = final_selected_df.loc[final_selected_df[TARGET_COLUMN] == 0, feature].dropna()
          x1 = final_selected_df.loc[final_selected_df[TARGET_COLUMN] == 1, feature].dropna()

          plt.figure(figsize=(8, 4))
          plt.hist(x0, bins=40, alpha=0.6, density=True, label="0 = sin anillo")
          plt.hist(x1, bins=40, alpha=0.6, density=True, label="1 = anillo interno")
          plt.title(f"Distribución por clase: {feature}")
          plt.xlabel(feature)
          plt.ylabel("Densidad aproximada")
          plt.legend()
          plt.tight_layout()

          feature_safe_name = re.sub(r"[^A-Za-z0-9_\-]+", "_", feature)
          plot_path = FEATURE_VIZ_DIR / f"distribution_{feature_safe_name}.png"
          plt.savefig(plot_path, dpi=150, bbox_inches="tight")
          plt.show()
  else:
      for feature in features_to_plot:
          plt.figure(figsize=(8, 4))
          plt.hist(final_selected_df[feature].dropna(), bins=40)
          plt.title(f"Distribución: {feature}")
          plt.xlabel(feature)
          plt.ylabel("Frecuencia")
          plt.tight_layout()

          feature_safe_name = re.sub(r"[^A-Za-z0-9_\-]+", "_", feature)
          plot_path = FEATURE_VIZ_DIR / f"distribution_{feature_safe_name}.png"
          plt.savefig(plot_path, dpi=150, bbox_inches="tight")
          plt.show()

  # Matriz de correlación
  corr_df = final_selected_df[selected_features].corr(method="spearman")

  plt.figure(figsize=(max(8, 0.45 * len(selected_features)), max(6, 0.45 * len(selected_features))))
  plt.imshow(corr_df, aspect="auto", vmin=-1, vmax=1)
  plt.colorbar(label="Correlación Spearman")
  plt.xticks(range(len(selected_features)), selected_features, rotation=90)
  plt.yticks(range(len(selected_features)), selected_features)
  plt.title("Correlación entre features seleccionadas")
  plt.tight_layout()

  corr_plot_path = FEATURE_VIZ_DIR / "selected_features_correlation.png"
  plt.savefig(corr_plot_path, dpi=150, bbox_inches="tight")
  plt.show()

  print("Graficas guardadas en:")
  print(FEATURE_VIZ_DIR)